In [ ]:
!pip install optuna mlflow importnb

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import time
import mlflow
import mlflow.pytorch
import scipy.signal
import torch.nn.functional as F
from google.colab import drive
import sys
import cv2
from importnb import Notebook

In [ ]:
file_path = ('/content/drive/MyDrive/2025_PPG_GLUC/Code/Model/Plots')

if file_path not in sys.path:
  sys.path.append(file_path)

# with Notebook():
with Notebook():
  import Eval_Metrics
  import Parkes_Error_Grid
from Eval_Metrics import plot_performance
from Parkes_Error_Grid import parkes_error_grid

In [ ]:
# Change paths below
DATA_DIR = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model Data/100Hz_15pulses/bin_concat/test_min_max_norm/'

# Output Directory
MODEL_SAVE_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Models/ResNet16_15bin_with_demographics/ACTUAL BEST/'
if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)

# MLflow
mlflow.set_tracking_uri("file:/content/drive/MyDrive/2025_PPG_GLUC/mlruns")

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
SEED = 42
INPUT_LENGTH = 1500

# Device
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Loading dataset

In [ ]:
class PPGDataset(Dataset):
    def __init__(self, X, y, M, d, augment=False):
        self.X = X
        self.y = y
        self.m = M
        self.d = d
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        m = self.m[idx].astype(np.float32)
        d = self.d[idx]

        # Augmentation
        if self.augment and np.random.rand() > 0.5:
            scale = np.random.uniform(0.9, 1.1)
            x = x.copy()
            x[m>0] *= scale
        x_t = torch.from_numpy(x).unsqueeze(0)
        m_t = torch.from_numpy(m).unsqueeze(0)
        y_t = torch.tensor(y, dtype=torch.float32)
        d_t = torch.from_numpy(d).float()

        return x_t, y_t, m_t, d_t

In [ ]:
def load_data():
    print("Loading data")
    # Change paths below
    X_train = np.load(os.path.join(DATA_DIR, 'X_train_pad_before_concat.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train_pad_before_concat.npy'))
    m_train = np.load(os.path.join(DATA_DIR, 'mask_train_pad_before_concat.npy'))
    d_train = np.load(os.path.join(DATA_DIR, 'demo_train_pad_before_concat.npy'))
    X_val = np.load(os.path.join(DATA_DIR, 'X_val_pad_before_concat.npy'))
    y_val = np.load(os.path.join(DATA_DIR, 'y_val_pad_before_concat.npy'))
    m_val = np.load(os.path.join(DATA_DIR, 'mask_val_pad_before_concat.npy'))
    d_val = np.load(os.path.join(DATA_DIR, 'demo_val_pad_before_concat.npy'))


    if os.path.exists(os.path.join(DATA_DIR, 'X_test_pad_before_concat.npy')):
        X_test = np.load(os.path.join(DATA_DIR, 'X_test_pad_before_concat.npy'))
        y_test = np.load(os.path.join(DATA_DIR, 'y_test_pad_before_concat.npy'))
        m_test = np.load(os.path.join(DATA_DIR, 'mask_test_pad_before_concat.npy'))
        d_test = np.load(os.path.join(DATA_DIR, 'demo_test_pad_before_concat.npy'))
    else:
        X_test, y_test = X_val, y_val
    print("Train sizes:", X_train.shape, y_train.shape, m_train.shape, d_train.shape)
    print("Val sizes:", X_val.shape, y_val.shape, m_val.shape, d_val.shape)
    print("Test sizes:", X_test.shape, y_test.shape, m_test.shape, d_test.shape)
    # Note: No manual scaling here because Dataset handles Z-score internally
    loaders = {
        'train': DataLoader(PPGDataset(X_train, y_train, m_train, d_train, augment=False), batch_size=BATCH_SIZE, shuffle=True),
        'val': DataLoader(PPGDataset(X_val, y_val, m_val, d_val, augment=False), batch_size=BATCH_SIZE, shuffle=False),
        'test': DataLoader(PPGDataset(X_test, y_test, m_test, d_test, augment=False), batch_size=BATCH_SIZE, shuffle=False)
    }
    return loaders

## Model Architecture

In [ ]:
class CNNBackbone(nn.Module):
  def __init__(self, in_channels=1, out_channels=1):
    super().__init__()
    self.conv1 = nn.Conv1d(in_channels, 64, kernel_size=7, stride=2, padding=3)
    self.bn1 = nn.BatchNorm1d(64)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv3 = nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1)
    self.bn3 = nn.BatchNorm1d(256)

    self.emb_dim = 256

  def forward(self, x):
    x = F.relu(self.bn1(self.conv1(x)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    return x

## Adding Masked GAP to ignore zero paddded areas


In [ ]:
def masked_gap_from_zero_padding(features, mask, eps=1e-6):
  """
  features: [B, C, T_feat]
  returns:  [B, C]
  """
  if mask.dim() == 2:
        mask = mask.unsqueeze(1)
  B, C, T_feat = features.shape
  mask_feat = F.interpolate(mask.float(), size=T_feat, mode="area")  # [B,1,T_feat]
  masked = features * mask_feat
  denom = mask_feat.sum(dim=-1).clamp(min=eps)  # [B,1]
  return masked.sum(dim=-1) / denom             # [B,C]

In [ ]:
class DemoMLP(nn.Module):
  def __init__(self, in_dim, emb_dim=64, dropout=0.2):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 64),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(64, emb_dim),
        nn.ReLU(),
    )
    self.emb_dim = emb_dim

  def forward(self, d):
    if d.dim() == 3:
      d = d.squeeze(1)
    return self.net(d)

In [ ]:
class MultiModalModel(nn.Module):
  def __init__(self, demo_dim=8, dropout=0.3):
    super().__init__()
    self.cnn = CNNBackbone()
    self.mlp = DemoMLP(demo_dim, emb_dim=256, dropout=dropout)
    fused_dim = self.cnn.emb_dim + self.mlp.emb_dim

    self.head = nn.Sequential(
        nn.Linear(fused_dim, 512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(64,1)
    )

  def forward(self, x, m=None, d=None):
    feats = self.cnn(x)
    sig_emb = masked_gap_from_zero_padding(feats, m)
    if d is None:
      fused = sig_emb
    else:
      demo_emb = self.mlp(d)
      fused = torch.cat([sig_emb, demo_emb], dim=1)

    out = self.head(fused).squeeze(-1)
    return out

## Training

In [ ]:
def plot_performance(model, loaders, history, best_val_mae, MODEL_SAVE_PATH, device):
    # Now the function 'knows' what model, loaders, and device are
    model.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x, y, lens, demos in loaders['test']:
            x, lens, demos = x.to(device), lens.to(device), demos.to(device)
            out = model(x, lens, demos)
            preds.extend(out.cpu().numpy())
            trues.extend(y.numpy())

    preds = np.array(preds).flatten()
    trues = np.array(trues).flatten()

    preds = np.array(preds); trues = np.array(trues)
    # 1. METRICS
    mae = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    mard = np.mean(np.abs((trues - preds) / trues)) * 100
    r2 = r2_score(trues, preds)

    print(f"\n📊 TEST RESULTS:")
    print(f"   RMSE: {rmse:.2f} mg/dL")
    print(f"   MAE:  {mae:.2f} mg/dL")
    print(f"   MARD: {mard:.2f} %")
    print(f"   R^2:  {r2:.4f}")

    # 2. Plotting
    plt.figure(figsize=(14, 6))

    # MAE Subplot
    plt.subplot(1, 2, 1)
    plt.plot(history['t_mae'], label='Train MAE', linewidth=2)
    plt.plot(history['v_mae'], label='Val MAE', linewidth=2)
    plt.plot(history['test_mae'], label='Test MAE', linestyle='--', alpha=0.7)
    plt.ylim(15,35)
    plt.title(f'MAE Learning Curve\n(Best Val MAE: {best_val_mae:.2f})')
    plt.xlabel('Epoch'); plt.ylabel('MAE (mg/dL)')
    plt.legend(); plt.grid(True, alpha=0.3)

    # Loss Subplot
    plt.subplot(1, 2, 2)
    plt.plot(history['t_loss'], label='Train Loss')
    plt.plot(history['v_loss'], label='Val Loss')
    plt.plot(history['test_loss'], label='Test Loss', linestyle='--')
    plt.ylim(15, 40)
    plt.title('Loss (MSE) Learning Curve')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    plt.savefig(os.path.join(MODEL_SAVE_PATH, 'learning_curves.png'))
    plt.show()
    return trues, preds

In [ ]:
def train_step(model, loader, crit, opt):
    model.train()
    loss_sum, mae_sum, count = 0, 0, 0
    for x, y, lens, demo in loader:
        x, y, lens, demo = x.to(device), y.to(device), lens.to(device), demo.to(device)
        opt.zero_grad()
        pred = model(x, lens, demo)
        loss = crit(pred, y)
        loss.backward()
        opt.step()
        loss_sum += loss.item()*x.size(0); mae_sum += (pred-y).abs().sum().item(); count += x.size(0)
    return loss_sum/count, mae_sum/count

def val_step(model, loader, crit):
    model.eval()
    loss_sum, mae_sum, count = 0, 0, 0
    with torch.no_grad():
        for x, y, lens, demo in loader:
            x, y, lens, demo = x.to(device), y.to(device), lens.to(device), demo.to(device)
            pred = model(x, lens, demo)
            loss_sum += crit(pred, y).item()*x.size(0); mae_sum += (pred-y).abs().sum().item(); count += x.size(0)
    return loss_sum/count, mae_sum/count

In [ ]:
def main():
    loaders = load_data()
    model = MultiModalModel().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.SmoothL1Loss(beta=10.0)

    best_path = os.path.join(MODEL_SAVE_PATH, 'best_model.pt')
    best_val_mae = float('inf')
    history = {'t_loss': [], 'v_loss': [], 'test_loss': [],
               't_mae': [], 'v_mae': [], 'test_mae': []}


    print(" Starting Training ")
    for epoch in range(EPOCHS):
        start = time.time()
        t_loss, t_mae = train_step(model, loaders['train'], criterion, optimizer)
        v_loss, v_mae = val_step(model, loaders['val'], criterion)
        test_loss, test_mae = val_step(model, loaders['test'], criterion)

        history['t_loss'].append(t_loss); history['t_mae'].append(t_mae)
        history['v_loss'].append(v_loss); history['v_mae'].append(v_mae)
        history['test_loss'].append(test_loss); history['test_mae'].append(test_mae)

        print(f"Ep {epoch+1}/{EPOCHS} | {time.time()-start:.0f}s | "
              f"T_MAE: {t_mae:.2f} | V_MAE: {v_mae:.2f} | Test_MAE: {test_mae:.2f}")


        if v_mae < best_val_mae:
            best_val_mae = v_mae
            torch.save(model.state_dict(), best_path)

    # --- EVALUATION ---
    print("\n--- Final Eval")

    model.load_state_dict(torch.load(best_path, map_location=device))

    trues, preds = plot_performance(
            model=model,
            loaders=loaders,
            history=history,
            best_val_mae=best_val_mae,
            MODEL_SAVE_PATH=MODEL_SAVE_PATH,
            device=device
        )

    mard = np.mean(np.abs((trues - preds) / trues)) * 100

    fig = parkes_error_grid(trues, preds, f"Parkes Grid (MARD: {mard:.1f}%)")
    fig.savefig(os.path.join(MODEL_SAVE_PATH, 'parkes_grid.png'))
    plt.show()


In [ ]:
if __name__ == "__main__":
  main()

## Calibration

In [ ]:
# Change paths below
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

DATA_DIR = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model Data/100Hz_15pulses/bin_concat/test_min_max_norm/'
MODEL_SAVE_PATH = '/content/drive/MyDrive/2025_PPG_GLUC/Models/ResNet16_15bin_with_demographics/ACTUAL BEST/'
META_CSV = '/content/drive/MyDrive/2025_PPG_GLUC/Data/Model/100Hz/demographics_15_bin_concat/100Hz_test_15pulses_demographics.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

In [ ]:
# Change paths here
X_test = np.load(os.path.join(DATA_DIR, 'X_test_pad_before_concat.npy'))
y_test = np.load(os.path.join(DATA_DIR, 'y_test_pad_before_concat.npy'))
m_test = np.load(os.path.join(DATA_DIR, 'mask_test_pad_before_concat.npy'))
d_test = np.load(os.path.join(DATA_DIR, 'demo_test_pad_before_concat.npy'))

test_loader = DataLoader(PPGDataset(X_test, y_test, m_test, d_test, augment=False),
                         batch_size=BATCH_SIZE, shuffle=False)

model = MultiModalModel().to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_SAVE_PATH, 'best_model.pt'),
                                 map_location=device))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for x, y, m, d in test_loader:
        x, m, d = x.to(device), m.to(device), d.to(device)
        preds.extend(model(x, m, d).cpu().numpy())
        trues.extend(y.numpy())

preds = np.array(preds).flatten()
trues = np.array(trues).flatten()
print("preds/trues:", preds.shape, trues.shape)

In [ ]:
meta = pd.read_csv(META_CSV)
assert len(meta) == len(preds), f"Row mismatch: meta={len(meta)} vs preds={len(preds)}"

df = pd.DataFrame({'caseid': meta['caseid'].values, 'win_id': meta['win_id'].values,
                   'y_true': trues, 'y_pred': preds})
df = df.sort_values(['caseid', 'win_id']).reset_index(drop=True)

N_CAL = 2  # first 2 windows per patient used for calibration

y_true_eval, y_pred_raw, y_pred_cal = [], [], []
skipped = 0
for cid, g in df.groupby('caseid', sort=False):
    if len(g) <= N_CAL:
        skipped += 1
        continue
    cal, ev = g.iloc[:N_CAL], g.iloc[N_CAL:]
    offset = cal['y_true'].mean() - cal['y_pred'].mean()
    y_true_eval.extend(ev['y_true'].values)
    y_pred_raw.extend(ev['y_pred'].values)
    y_pred_cal.extend(ev['y_pred'].values + offset)

y_true_eval = np.array(y_true_eval)
y_pred_raw  = np.array(y_pred_raw)
y_pred_cal  = np.array(y_pred_cal)

print(f"Patients used: {df['caseid'].nunique() - skipped} | skipped (<={N_CAL} wins): {skipped}")
print(f"Eval samples:  {len(y_true_eval)}")

def report(name, y, yhat):
    pr, _ = pearsonr(y, yhat)
    sr, _ = spearmanr(y, yhat)
    mae = np.mean(np.abs(y - yhat))
    print(f"{name:12s} | Pearson r: {pr:.4f} | Spearman: {sr:.4f} | MAE: {mae:.2f}")

print("\n  Correlation: actual vs predicted (windows 3+ only) ")
report("Before cal", y_true_eval, y_pred_raw)
report("After cal",  y_true_eval, y_pred_cal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
lims = [min(y_true_eval.min(), y_pred_raw.min(), y_pred_cal.min()) - 5,
        max(y_true_eval.max(), y_pred_raw.max(), y_pred_cal.max()) + 5]

for ax, yhat, title in [(axes[0], y_pred_raw, 'Before calibration'),
                        (axes[1], y_pred_cal, 'After calibration')]:
    ax.scatter(y_true_eval, yhat, alpha=0.3, s=10)
    ax.plot(lims, lims, 'r--', lw=1)
    r, _ = pearsonr(y_true_eval, yhat)
    ax.set_title(f'{title}  (r = {r:.3f})')
    ax.set_xlabel('Actual glucose (mg/dL)'); ax.set_ylabel('Predicted glucose (mg/dL)')
    ax.set_xlim(lims); ax.set_ylim(lims); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_PATH, 'calibration_scatter.png'), dpi=120)
plt.show()

In [ ]:
rows = []
for cid, g in df.groupby('caseid', sort=False):
    if len(g) <= N_CAL:
        continue
    cal, ev = g.iloc[:N_CAL], g.iloc[N_CAL:]
    offset = cal['y_true'].mean() - cal['y_pred'].mean()

    yt = ev['y_true'].values
    yp_raw = ev['y_pred'].values
    yp_cal = yp_raw + offset

    # Per-patient correlation needs variance in y_true; if constant, skip r
    def safe_pearson(a, b):
        if np.std(a) == 0 or np.std(b) == 0 or len(a) < 2:
            return np.nan
        return pearsonr(a, b)[0]

    rows.append({
        'caseid': cid,
        'n_windows_total': len(g),
        'n_eval': len(ev),
        'offset': offset,
        'y_true_mean': yt.mean(),
        'y_true_std':  yt.std(),
        'mae_before': np.mean(np.abs(yt - yp_raw)),
        'mae_after':  np.mean(np.abs(yt - yp_cal)),
        'pearson_before':  safe_pearson(yt, yp_raw),
        'pearson_after':   safe_pearson(yt, yp_cal),  # same as before (offset doesn't change r within a patient)
    })

per_patient = pd.DataFrame(rows)
out_path = os.path.join(MODEL_SAVE_PATH, 'calibration_per_patient.csv')
per_patient.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\nPer-patient summary (across {len(per_patient)} patients):")
print(per_patient[['mae_before','mae_after','pearson_before','pearson_after']].describe().round(3))

# Mean of per-patient MAEs (patient-level average, not sample-level)
print(f"\nMean per-patient MAE  | before: {per_patient['mae_before'].mean():.2f}"
      f" | after: {per_patient['mae_after'].mean():.2f}")
print(f"Median per-patient MAE| before: {per_patient['mae_before'].median():.2f}"
      f" | after: {per_patient['mae_after'].median():.2f}")